## Goal

Build an auditable frozen acoustic dataset through the reviewed pipeline. The
default is `SMOKE_MODE`; private-data and network processing require an explicit
authorised full-run environment. Reader-visible results are aggregate or
redacted.

## Setup

Resolve the repository, read the source/runtime parameters, and run the local
preflight. This notebook never installs packages or downloads model weights.
`FORCE_STAGE` additionally requires `THESIS_FORCE_STAGE` to name acquisition,
audio, ASR, features, or all. The one-time legacy-cache migration is opt-in via
`THESIS_ALLOW_LEGACY_STAGE_CACHE=1`.

In [ ]:
import os
from pathlib import Path
from presentation_assessment.research.notebook_support import (
    bounded_feature_preview,
    display_aggregate_payload,
    execute_notebook_pipeline,
    freeze_notebook_dataset,
    locate_repository_root,
    prepare_notebook_run,
    redacted_freeze_summary,
    stage_summary,
    summarize_preflight,
    summarize_quality,
    summarize_source_audit,
    validate_notebook_outputs,
    write_redacted_handoff,
)

repository_root = locate_repository_root(Path.cwd())
notebook_mode = os.environ.get("THESIS_NOTEBOOK_MODE", "SMOKE_MODE")
network_opt_in = os.environ.get("THESIS_ALLOW_YOUTUBE_NETWORK", "0")
dataset_config = os.environ.get("THESIS_DATASET_CONFIG", "configs/notebook_pipeline.yaml")
source_mode = os.environ.get("THESIS_SOURCE_MODE", "YOUTUBE")
local_source_config = os.environ.get("THESIS_LOCAL_SOURCE_CONFIG", "")
run_context = prepare_notebook_run(
    repository_root,
    mode=notebook_mode,
    network_opt_in=network_opt_in,
    config_path=dataset_config,
    source_mode=source_mode,
    local_source_config=local_source_config or None,
)
display_aggregate_payload(summarize_preflight(run_context))

## Source Audit

Confirm that every intended row is countable before any acquisition starts.
Only aggregate availability and verification counts are displayed.

In [ ]:
source_audit = summarize_source_audit(run_context)
display_aggregate_payload(source_audit)

## Acquisition

Run the tested resumable orchestration. Smoke mode is synthetic; authorised
full runs use either explicit YouTube acquisition or validated local files.

In [ ]:
pipeline_result = execute_notebook_pipeline(run_context)
display_aggregate_payload(stage_summary(pipeline_result, "acquisition"))

## Audio

Report terminal audio-stage counts. Failed inputs remain represented by an
explicit terminal state.

In [ ]:
display_aggregate_payload(stage_summary(pipeline_result, "audio"))

## Whisper

Report local transcription-stage counts. Full runs bind the exact validated
snapshot and reuse one model instance for the batch.

In [ ]:
display_aggregate_payload(stage_summary(pipeline_result, "asr"))

## Features

Report terminal feature-stage counts and a bounded aggregate availability
preview. No sample identifiers or feature values are displayed.

In [ ]:
display_aggregate_payload(stage_summary(pipeline_result, "features"))
feature_preview = bounded_feature_preview(pipeline_result)
display_aggregate_payload(feature_preview)

## Quality

Summarize feature availability, missingness, and warnings without exposing
row-level source material.

In [ ]:
quality_summary = summarize_quality(pipeline_result)
display_aggregate_payload(quality_summary)

## Freeze

Create or reuse the content-addressed immutable dataset and show only its
public-safe identity and aggregate terminal counts.

In [ ]:
frozen_dataset = freeze_notebook_dataset(pipeline_result)
display_aggregate_payload(redacted_freeze_summary(frozen_dataset, pipeline_result))

## Checks

Verify row completeness, required terminal records, frozen hashes, and the
privacy boundary before handoff.

In [ ]:
validation = validate_notebook_outputs(run_context, pipeline_result, frozen_dataset)
display_aggregate_payload(validation)

## Next Steps

Write a redacted local handoff summary for the next notebook. The executed full
copy and all row-level artifacts stay inside ignored research artifacts.

In [ ]:
if validation.get("valid") is not True:
    raise RuntimeError("Strict frozen-dataset validation failed")
handoff = write_redacted_handoff(run_context, pipeline_result, frozen_dataset, validation)
display_aggregate_payload(handoff)